In [1]:
import tkinter as tk
import random
import math
TRANG_THAI_BAN_DAU = [
    [1, 2, 3],
    [4, 5, 6],
    [0, 7, 8]
]

KET_QUA_DICH = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 0]
]

def lam_phang_ma_tran(ma_tran):
    mảng_phẳng = []
    for hang in ma_tran:
        mảng_phẳng.extend(hang)
    return mảng_phẳng

TRANG_THAI_DICH_PHANG = tuple(lam_phang_ma_tran(KET_QUA_DICH))
TRANG_THAI_DAU_PHANG = lam_phang_ma_tran(TRANG_THAI_BAN_DAU)

CAC_HUONG = ["Trai", "Phai", "Len", "Xuong"]
HUONG_NGUOC = {"Trai": "Phai", "Phai": "Trai", "Len": "Xuong", "Xuong": "Len"}

class GiaoDienSimulatedAnnealing:
    def __init__(self, cua_so):
        self.cua_so = cua_so
        self.cua_so.title("8-Puzzle - Simulated Annealing Simulator")

        self.ban_co = list(TRANG_THAI_DAU_PHANG)
        self.so_buoc = 0
        self.huong_vua_di = "Bắt đầu"
        self.dang_tu_dong = False
        self._after_id = None
        
        self.vi_tri_dong_cac_buoc = {}

        self.vi_tri_dich_goc = {}
        for idx, s in enumerate(TRANG_THAI_DICH_PHANG):
            if s != 0:
                self.vi_tri_dich_goc[s] = divmod(idx, 3)
        khung_tren = tk.Frame(cua_so)
        khung_tren.pack(pady=10, fill="x")

        self.nhan_trang_thai = tk.Label(
            khung_tren,
            text="Số bước: 0",
            anchor="w",
            font=("Arial", 11)
        )
        self.nhan_trang_thai.pack(side="left", padx=10, fill="x", expand=True)

        tk.Button(khung_tren, text="Chạy Simulated Annealing", command=self.giai_thuat_toan, bg="#ff4500", fg="white", font=("Arial", 10, "bold")).pack(side="right", padx=5)
        tk.Button(khung_tren, text="Dừng Auto", command=self.dung_tu_dong).pack(side="right", padx=5)
        tk.Button(khung_tren, text="Trộn Bàn Cờ Sâu", command=lambda: self.tron(40)).pack(side="right", padx=5)
        tk.Button(khung_tren, text="Đặt Lại", command=self.dat_lai).pack(side="right", padx=5)
        khung_chinh = tk.Frame(cua_so)
        khung_chinh.pack(padx=10, pady=10, fill="both", expand=True)
        khung_left = tk.Frame(khung_chinh)
        khung_left.pack(side="left", anchor="n")

        khung_ban_co = tk.Frame(khung_left, bg="darkgoldenrod2", padx=8, pady=8)
        khung_ban_co.pack(anchor="n")

        self.danh_sach_o = []
        for hang in range(3):
            for cot in range(3):
                o = tk.Label(
                    khung_ban_co,
                    text="",
                    width=4,
                    height=2,
                    font=("Arial", 22, "bold"),
                    bg="white",
                    relief="raised",
                    borderwidth=3
                )
                o.grid(row=hang, column=cot, padx=4, pady=4)
                self.danh_sach_o.append(o)

        # Khối 2: Trạng thái đích (Goal State) ở giữa cố định
        khung_dich = tk.Frame(khung_chinh, bg="#888", padx=10, pady=10)
        khung_dich.pack(side="left", padx=20, anchor="n")

        tk.Label(khung_dich, text="GOAL STATE", font=("Arial", 12, "bold"), bg="#888", fg="white").pack(pady=5)

        luoi_dich = tk.Frame(khung_dich, bg="#888")
        luoi_dich.pack()

        for i, gia_tri in enumerate(TRANG_THAI_DICH_PHANG):
            hang, cot = divmod(i, 3)
            chu = "" if gia_tri == 0 else str(gia_tri)
            mau_nen = "#bbb" if gia_tri != 0 else "#999"

            o_dich = tk.Label(
                luoi_dich,
                text=chu,
                width=4,
                height=2,
                font=("Arial", 16, "bold"),
                bg=mau_nen,
                relief="ridge",
                borderwidth=2
            )
            o_dich.grid(row=hang, column=cot, padx=3, pady=3)
        khung_phai = tk.Frame(khung_chinh)
        khung_phai.pack(side="left", padx=10, fill="both", expand=True)

        tk.Label(khung_phai, text="QUÁ TRÌNH LUYỆN KIM (ANNEALING LOG)", font=("Arial", 10, "bold"), fg="#ff4500").pack(anchor="w", pady=2)
        
        thanh_cuon = tk.Scrollbar(khung_phai)
        thanh_cuon.pack(side="right", fill="y")

        self.o_chu_log = tk.Text(
            khung_phai,
            width=42,
            height=16,
            font=("Courier", 10, "bold"),
            bg="#fdfdfd",
            yscrollcommand=thanh_cuon.set
        )
        self.o_chu_log.pack(side="left", fill="both", expand=True)
        thanh_cuon.config(command=self.o_chu_log.yview)
        self.o_chu_log.tag_config("trang_thai_cu", foreground="#999999", background="")
        self.o_chu_log.tag_config("highlight_dong", foreground="white", background="#ff4500")

        cua_so.bind("<Left>",  lambda e: self.xu_ly_phim("Trai"))
        cua_so.bind("<Right>", lambda e: self.xu_ly_phim("Phai"))
        cua_so.bind("<Up>",    lambda e: self.xu_ly_phim("Len"))
        cua_so.bind("<Down>",  lambda e: self.xu_ly_phim("Xuong"))

        self.ve_giao_dien()

    def _tinh_manhattan(self, trang_thai):
        tong_khoang_cach = 0
        for i, so in enumerate(trang_thai):
            if so != 0:
                r_hien_tai, c_hien_tai = divmod(i, 3)
                r_dich, c_dich = self.vi_tri_dich_goc[so]
                tong_khoang_cach += abs(r_hien_tai - r_dich) + abs(c_hien_tai - c_dich)
        return tong_khoang_cach

    def nap_truoc_toan_bo_hanh_trinh_vao_log(self, hanh_trinh, bat_dau):
        """In trước toàn bộ cây lịch sử tối ưu hóa cấu trúc hạt vào Text Log"""
        self.o_chu_log.delete("1.0", tk.END)
        self.vi_tri_dong_cac_buoc.clear()
        
        buoc_tam = 0
        self._ghi_mot_khoi_ma_tran_vao_text(buoc_tam, "Bắt đầu", 25.0, 1.0, "Khởi tạo", bat_dau)
        
        for kieu_hanh_dong, chi_tiet, t_nhiet, xac_suat, mang_cau_hinh in hanh_trinh:
            buoc_tam += 1
            ghi_chu = "Leo đồi tốt" if kieu_hanh_dong == "TOT_HON" else "Vượt bẫy xấu"
            self._ghi_mot_khoi_ma_tran_vao_text(buoc_tam, chi_tiet, t_nhiet, xac_suat, ghi_chu, mang_cau_hinh)
            
        self.o_chu_log.tag_add("trang_thai_cu", "1.0", tk.END)

    def _ghi_mot_khoi_ma_tran_vao_text(self, buoc, huong, temp, prob, ghi_chu, mang):
        h_manhattan = self._tinh_manhattan(tuple(mang))
        m = [str(x) if x != 0 else " " for x in mang]
        
        danh_dau_dau = self.o_chu_log.index(tk.END + "-1c")
        
        van_ban_ma_tran = (
            f"┌─ Bước {buoc:03d} ────────────────────────┐\n"
            f"│ Hướng đi: {huong:<7} | Loại: {ghi_chu:<11} │\n"
            f"│ Nhiệt độ T: {temp:<7.2f} | P_nhận: {prob:<7.3f} │\n"
            f"│ Manhattan: {h_manhattan:<29} │\n"
            f"│   [ {m[0]} {m[1]} {m[2]} ]                                │\n"
            f"│   [ {m[3]} {m[4]} {m[5]} ]                                │\n"
            f"│   [ {m[6]} {m[7]} {m[8]} ]                                │\n"
            f"└──────────────────────────────────────┘\n"
            f"                   ▼                    \n"
        )
        self.o_chu_log.insert(tk.END, van_ban_ma_tran)
        danh_dau_cuoi = self.o_chu_log.index(tk.END + "-1c")
        
        self.vi_tri_dong_cac_buoc[buoc] = (danh_dau_dau, danh_dau_cuoi)

    def ve_giao_dien(self):
        for i, gia_tri in enumerate(self.ban_co):
            o = self.danh_sach_o[i]
            if gia_tri == 0:
                o.config(text="", bg="#555", relief="sunken")
            else:
                o.config(text=str(gia_tri), bg="#f2f2f2", relief="raised")

        if tuple(self.ban_co) == TRANG_THAI_DICH_PHANG:
            self.nhan_trang_thai.config(text=f"Chúc mừng! Đã hạ nhiệt cấu trúc tinh thể đạt đích bền vững.", fg="green")
        else:
            h_manhattan = self._tinh_manhattan(tuple(self.ban_co))
            self.nhan_trang_thai.config(text=f"Số bước mô phỏng: {self.so_buoc} | Khoảng cách Manhattan (h): {h_manhattan}", fg="black")
        if self.so_buoc in self.vi_tri_dong_cac_buoc:
            self.o_chu_log.tag_remove("highlight_dong", "1.0", tk.END)
            dong_dau, dong_cuoi = self.vi_tri_dong_cac_buoc[self.so_buoc]
            self.o_chu_log.tag_add("highlight_dong", dong_dau, dong_cuoi)
            self.o_chu_log.see(dong_dau)
        else:
            self.o_chu_log.tag_remove("highlight_dong", "1.0", tk.END)
            m = [str(x) if x != 0 else " " for x in self.ban_co]
            h_manhattan = self._tinh_manhattan(tuple(self.ban_co))
            dau = self.o_chu_log.index(tk.END + "-1c")
            van_ban = (
                f"┌─ Bước {self.so_buoc:03d} (Thủ công) ──────────┐\n"
                f"│ Hướng đi phím: {self.huong_vua_di:<21} │\n"
                f"│ Manhattan hiện tại: {h_manhattan:<16} │\n"
                f"│   [ {m[0]} {m[1]} {m[2]} ]                                │\n"
                f"│   [ {m[3]} {m[4]} {m[5]} ]                                │\n"
                f"│   [ {m[6]} {m[7]} {m[8]} ]                                │\n"
                f"└──────────────────────────────────────┘\n"
                f"                   ▼                    \n"
            )
            self.o_chu_log.insert(tk.END, van_ban)
            cuoi = self.o_chu_log.index(tk.END + "-1c")
            self.o_chu_log.tag_add("highlight_dong", dau, cuoi)
            self.o_chu_log.see(tk.END)

    def dat_lai(self):
        self.dung_tu_dong()
        self.ban_co = list(TRANG_THAI_DAU_PHANG)
        self.so_buoc = 0
        self.huong_vua_di = "Bắt đầu"
        self.o_chu_log.delete("1.0", tk.END)
        self.vi_tri_dong_cac_buoc.clear()
        self.ve_giao_dien()

    def xu_ly_phim(self, huong):
        if self.dang_tu_dong or tuple(self.ban_co) == TRANG_THAI_DICH_PHANG:
            return
        if self.di_chuyen_o_trong(self.ban_co, huong):
            self.so_buoc += 1
            self.huong_vua_di = huong
            self.ve_giao_dien()

    def di_chuyen_o_trong(self, mang_ban_co, huong):
        vi_tri_trong = mang_ban_co.index(0)
        hang, cot = divmod(vi_tri_trong, 3)

        if huong == "Trai": hang_moi, cot_moi = hang, cot - 1
        elif huong == "Phai": hang_moi, cot_moi = hang, cot + 1
        elif huong == "Len": hang_moi, cot_moi = hang - 1, cot
        elif huong == "Xuong": hang_moi, cot_moi = hang + 1, cot
        else: return False

        if not (0 <= hang_moi < 3 and 0 <= cot_moi < 3):
            return False

        vi_tri_moi = hang_moi * 3 + cot_moi
        mang_ban_co[vi_tri_trong], mang_ban_co[vi_tri_moi] = mang_ban_co[vi_tri_moi], mang_ban_co[vi_tri_trong]
        return True

    def tron(self, so_buoc_tron=40):
        self.dung_tu_dong()
        self.ban_co = list(TRANG_THAI_DICH_PHANG)
        self.so_buoc = 0
        self.huong_vua_di = "Trộn"
        self.o_chu_log.delete("1.0", tk.END)
        self.vi_tri_dong_cac_buoc.clear()
        huong_truoc = None
        cac_huong = ["Trai", "Phai", "Len", "Xuong"]
        
        for _ in range(so_buoc_tron):
            random.shuffle(cac_huong)
            for huong in cac_huong:
                if huong_truoc and huong == HUONG_NGUOC[huong_truoc]:
                    continue
                if self.di_chuyen_o_trong(self.ban_co, huong):
                    huong_truoc = huong
                    break
        self.ve_giao_dien()

    def _hang_xom(self, trang_thai):
        idx0 = trang_thai.index(0)
        r, c = divmod(idx0, 3)

        def swap(t, i, j):
            lst = list(t)
            lst[i], lst[j] = lst[j], lst[i]
            return tuple(lst)

        ket_qua = []
        for huong in CAC_HUONG:
            if huong == "Trai": r2, c2 = r, c - 1
            elif huong == "Phai": r2, c2 = r, c + 1
            elif huong == "Len": r2, c2 = r - 1, c
            else: r2, c2 = r + 1, c

            if 0 <= r2 < 3 and 0 <= c2 < 3:
                idx2 = r2 * 3 + c2
                ket_qua.append((swap(trang_thai, idx0, idx2), huong))
        return ket_qua

    def thuat_toan_simulated_annealing(self, bat_dau):
        """Lõi giải thuật Simulated Annealing - Tối ưu hóa năng lượng hệ thống"""
        trang_thai_hien_tai = bat_dau
        chuoi_hanh_trinh = []
        T = 25.0              
        T_MIN = 0.05        
        ALPHA = 0.94          
        VONG_LAP_MAX = 800    

        vong_lap = 0
        while T > T_MIN and vong_lap < VONG_LAP_MAX:
            if trang_thai_hien_tai == TRANG_THAI_DICH_PHANG:
                return chuoi_hanh_trinh, "THANH_CONG", f"Thành công! Cấu trúc đạt đích ổn định tối ưu."
            cac_hang_xom = self._hang_xom(trang_thai_hien_tai)
            trang_thai_ke, huong_di = random.choice(cac_hang_xom)

            cost_hien_tai = self._tinh_manhattan(trang_thai_hien_tai)
            cost_ke = self._tinh_manhattan(trang_thai_ke)
            delta_E = cost_hien_tai - cost_ke 

            if delta_E > 0:
                trang_thai_hien_tai = trang_thai_ke
                chuoi_hanh_trinh.append(("TOT_HON", huong_di, T, 1.0, trang_thai_hien_tai))
            else:
                xac_suat_nhan = math.exp(delta_E / T)
                
                if random.random() < xac_suat_nhan:
                    trang_thai_hien_tai = trang_thai_ke
                    chuoi_hanh_trinh.append(("TE_HON_NHUNG_NHAN", huong_di, T, xac_suat_nhan, trang_thai_hien_tai))

            T *= ALPHA
            vong_lap += 1

        if trang_thai_hien_tai == TRANG_THAI_DICH_PHANG:
            return chuoi_hanh_trinh, "THANH_CONG", f"Thành công! Cấu trúc đạt đích ổn định tối ưu."
        return chuoi_hanh_trinh, "THAT_BAI", f"Kẹt đỉnh đóng băng! Nhiệt độ chạm đáy mà không ra đích."

    def chay_loi_giai(self, hanh_trinh, ket_qua_tim, thong_bao_giai, delay_ms=250):
        self.dang_tu_dong = True
        
        def step(i):
            if not self.dang_tu_dong: return
            if i >= len(hanh_trinh):
                self.dang_tu_dong = False
                self._after_id = None
                self.ve_giao_dien()
                
                if ket_qua_tim == "THANH_CONG":
                    self.nhan_trang_thai.config(text=thong_bao_giai, fg="green")
                else:
                    self.nhan_trang_thai.config(text=thong_bao_giai, fg="red")
                return

            kieu_hanh_dong, chi_tiet, t_nhiet, xac_suat, mang_cau_hinh = hanh_trinh[i]
            
            self.ban_co = list(mang_cau_hinh)
            self.so_buoc += 1
            self.huong_vua_di = chi_tiet
            
            if kieu_hanh_dong == "TOT_HON":
                self.nhan_trang_thai.config(text=f"Leo đồi tối ưu dốc đứng (T={t_nhiet:.2f})", fg="blue")
            else:
                self.nhan_trang_thai.config(text=f"Vượt bẫy cực cục bộ! (P={xac_suat:.2f} | T={t_nhiet:.2f})", fg="#d2691e")

            self.ve_giao_dien()
            self._after_id = self.cua_so.after(delay_ms, lambda: step(i + 1))
            
        step(0)

    def dung_tu_dong(self):
        self.dang_tu_dong = False
        if self._after_id is not None:
            try: self.cua_so.after_cancel(self._after_id)
            except Exception: pass
            self._after_id = None

    def giai_thuat_toan(self):
        if self.dang_tu_dong: return
        bat_dau = tuple(self.ban_co)
        self.nhan_trang_thai.config(text="Đang nung nhiệt tính toán hành trình Simulated Annealing...", fg="purple")
        self.cua_so.update_idletasks()

        hanh_trinh, ket_qua_tim, thong_bao_giai = self.thuat_toan_simulated_annealing(bat_dau)
        
        self.so_buoc = 0
        self.huong_vua_di = "Bắt đầu"
        
        # Nạp trước toàn bộ cây dữ liệu mô phỏng sang log bên phải
        self.nap_truoc_toan_bo_hanh_trinh_vao_log(hanh_trinh, bat_dau)
        
        # Chạy hoạt ảnh trực quan trên ma trận tương tác
        self.chay_loi_giai(hanh_trinh, ket_qua_tim, thong_bao_giai)

if __name__ == "__main__":
    cua_so_chinh = tk.Tk()
    GiaoDienSimulatedAnnealing(cua_so_chinh)
    cua_so_chinh.mainloop()